# Part 2: Diabetes

In this part of the assignment, you will build a predictive model for diabetes disease progression in the next year based on current observed features of disease symptoms. 

**Learning objectives.** You will:
1. Train and test a linear model using ordinary least squares regression.
2. Use numerical Python (NumPy) and the standard `sklearn` API in Python  
3. Train and test a quadratic model, comparing to the linear model and demonstrating overfitting
4. Apply regularization, specifically LASSO, to build a sparse linear model

The following code will download and preview three examples of the data. The ten features are as follows (in order):

- age age in years
- sex
- bmi body mass index
- bp average blood pressure
- s1 tc, total serum cholesterol
- s2 ldl, low-density lipoproteins
- s3 hdl, high-density lipoproteins
- s4 tch, total cholesterol / HDL
- s5 ltg, log of serum triglycerides level
- s6 glu, blood sugar level

The target value is a quantiative measure of disease progression after 1 year, where larger numbers are worse.

The code stores the feature matrix `X` as a two-dimensional NumPy array where each row corresponds to a data point and each column is a feature. The target value is stored as a one-dimensional NumPy array `y` where the index `i` element of `y` correpsonds to the row `i` data point of `X`.

Your overall goal in this part is to build and evaluate a linear model to predict the target variable `y` as a function of the ten features in `X`, and to identify which features are more significant for predicting `y`.

In [6]:
# Run but DO NOT MODIFY this code

from sklearn.datasets import load_diabetes

# Load the diabetes dataset
diabetes = load_diabetes(scaled = False)
print(diabetes.feature_names)

# Get the feature data and target variable
X = diabetes.data
y = diabetes.target

# Preview the first 3 data points
print(X[:3])
print(y[:3])

['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
[[ 59.       2.      32.1    101.     157.      93.2     38.       4.
    4.8598  87.    ]
 [ 48.       1.      21.6     87.     183.     103.2     70.       3.
    3.8918  69.    ]
 [ 72.       2.      30.5     93.     156.      93.6     41.       4.
    4.6728  85.    ]]
[151.  75. 141.]


## Task 1

Use `sklearn` to randomly split the input data into a [train and test partition](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), with 30% of the data reserved for testing. Use a random seed of `2025` for reproducibility of the results.

Print the number of data points in the resulting train and test partitions.

In [7]:
# Write task 1 code here
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=2025)

print("Number of training data points:", X_train.shape[0])
print("Number of testing data points:", X_test.shape[0])

Number of training data points: 309
Number of testing data points: 133


## Task 2

Build a baseline prediction by computing the [average](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) target value of the training data and predicting this for average for every test data point.

For example, if the training data target values were `[2, 2, 5]` then you would compute the average as `3`. If there there were only two test data points, then your predictions would be simply `[3, 3]`.

Evaluate the [root mean squared error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html#sklearn.metrics.root_mean_squared_error) between the baseline and the test data.

This RMSE will serve as a benchmark - any useful model should achieve a lower error than this simple baseline.

In [8]:
# Write task 2 code here
import numpy as np
from sklearn.metrics import root_mean_squared_error

y_train_mean = np.mean(y_train)

y_baseline_pred = np.full(shape=y_test.shape, fill_value=y_train_mean)

baseline_rmse = root_mean_squared_error(y_test, y_baseline_pred)

print("Baseline RMSE:", baseline_rmse)


Baseline RMSE: 75.8165287907097


## Task 3

Use [`sklearn`](https://scikit-learn.org/stable/) to fit a linear predictive model on the training data using [ordinary least squares regression](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares). 

Evaluate the [root mean squared error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html#sklearn.metrics.root_mean_squared_error) of the model on **both** the training data **and** the test data (that is, the training error and the generalization error). Report both results.

Note that the model predictions on the test data may not be perfect, but they should improve meaningfully over the simple baseline from Task 2 or something is wrong.

In [9]:
# Write task 3 code here
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

y_train_pred = lin_model.predict(X_train)
y_test_pred = lin_model.predict(X_test)

train_rmse = root_mean_squared_error(y_train, y_train_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)

print("Linear Regression (OLS) Training RMSE:", train_rmse)
print("Linear Regression (OLS) Test RMSE:", test_rmse)


Linear Regression (OLS) Training RMSE: 52.97771955290354
Linear Regression (OLS) Test RMSE: 55.14187488833097


## Task 4

To understand which input features are most important for predicting diabetes progression, we need a model that can automatically select relevant features. The ordinary least squares model from Task 3 uses all features, making it harder to identify which ones truly matter.

Build a new linear model using [Lasso regression](https://scikit-learn.org/stable/modules/linear_model.html#lasso) that meets two criteria:
  - Performance: At most 10% greater error than the linear model with all the features in task 3. 
  - Sparsity: At least three model coefficients set to 0 (meaning the model does not use these features to make predictions). You can treat any coefficient less than 0.0001 as effectively 0 for this task.

You may need to try multiple vaues of the `alpha` *hyperparameter* to find a satisfy both constraints. For example, you can try [0.1, 1, 5, 10, ...]. The final LASSO model is only required to satisfy the above two criteria. Nevertheless, you should only evaluate error on the test dataset **once** after searching for such a value of `alpha`. Use [cross validation](https://scikit-learn.org/stable/modules/cross_validation.html) on the training data or split the training data into train and validation sets.

For your final Lasso model with the chosen `alpha` fit on all of the training data, report the [root mean squared error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html#sklearn.metrics.root_mean_squared_error) of the model predictions on the test data. Report the three or more features for which the model coefficients were set to 0 (see feature names/interpretations above). Also please explain why you selected this `alpha`.

In [10]:
# Write task 4 code here
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error

feature_names = diabetes.feature_names

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=2025
)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)

alphas = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1, 3]

best_alpha = None
best_val_rmse = float("inf")

for a in alphas:
    model = Lasso(alpha=a, max_iter=20000, random_state=2025)
    model.fit(X_tr_s, y_tr)

    y_val_pred = model.predict(X_val_s)
    val_rmse = root_mean_squared_error(y_val, y_val_pred)

    num_zeros = np.sum(np.abs(model.coef_) < 1e-4)

    if num_zeros >= 3 and val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_alpha = a

print("Chosen alpha (from train/val):", best_alpha)
print("Validation RMSE for chosen alpha:", best_val_rmse)

scaler_final = StandardScaler()
X_train_s = scaler_final.fit_transform(X_train)
X_test_s = scaler_final.transform(X_test)

final_lasso = Lasso(alpha=best_alpha, max_iter=20000, random_state=2025)
final_lasso.fit(X_train_s, y_train)

y_test_pred = final_lasso.predict(X_test_s)
lasso_test_rmse = root_mean_squared_error(y_test, y_test_pred)

zero_features = [
    feature_names[i]
    for i, c in enumerate(final_lasso.coef_)
    if abs(c) < 1e-4
]

print("Final LASSO test RMSE:", lasso_test_rmse)
print("Number of zero coefficients:", len(zero_features))
print("Zeroed-out features:", zero_features)

print("110% of OLS test RMSE:", 1.10 * test_rmse)
print("Passes 110% rule:", lasso_test_rmse <= 1.10 * test_rmse)


Chosen alpha (from train/val): 3
Validation RMSE for chosen alpha: 57.59722407076614
Final LASSO test RMSE: 55.33715151797645
Number of zero coefficients: 4
Zeroed-out features: ['age', 's2', 's4', 's6']
110% of OLS test RMSE: 60.65606237716407
Passes 110% rule: True


*Report features and explain for task 4 here*

This alpha was chosen because the RMSE during validation for this alpha was minimised at 3 out of all the tested values of alpha. This also proves to be effective as in the final LASSO RMSE test, it reduces RMSE even more and it also performs more than 10% better than the linear model.